In [ ]:
#IMPORTS
'''import DataHandling as DH
import matplotlib.pyplot as plt
import utils as utl
import numpy as np
import Preprocessing as pre'''
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import DataHandling as DH
import Preprocessing as pre
import utils as utl
from datetime import datetime, timedelta
import os

# Set the base directory where JSON files are located
base_directory = "C:\\Users\\OCD\\Desktop\\DATA\\RN"  # Change this to your folder path

file_path = "C:\\Users\\OCD\\Desktop\\DATA\\RN\\2024-11-11\\Report_Json_Session_Report_20241125T163059.json"

Data = DH.getData(file_path)

### Timeline Sobreposta

In [ ]:
timeline = DH.getSignals(Data, 'Timeline')

arrays = []
for time in timeline:
    date = time['Left']['X'][-1]
    x = []

    for val in time['Left']['X']:
        date = utl.parse_datetime(val)
        new_date = datetime(year=1900, month=1, day=1, hour=date.hour, minute=date.minute)
        x.append(new_date)

    arrays.append(x)

max = -1
flag = -1
for i, xx in enumerate(arrays):
    if len(xx)>max: 
        max == len(xx)
        flag = i

x = arrays[flag]
plt.figure()
for time in timeline:
    for side in ['Left', 'Right']:
        sign = time[side]
        y = [float(val) for val in sign['Y']]
        if len(y)<len(x): 
            x = x[:len(y)]
        else: y = y[:len(x)]
        plt.plot(x,y,label = f"{time['Date']}-{side}")

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format x-axis labels as HH:MM
plt.xticks(rotation=45)  # Optional: Rotate labels for better readability
plt.legend()



plt.figure()
for time in timeline:
    for side in ['Left', 'Right']:
        sign = time[side]
        y = [float(val) for val in sign['Y']]
        x = [val for val in sign['X']]
        plt.plot(x,y,label = f"{time['Date']}-{side}")
        print(f"{time['Date']} || {x[0]} || {x[-1]}")
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))  # Format x-axis labels as HH:MM
        plt.xticks(rotation=45)  # Optional: Rotate labels for better readability


plt.legend()
plt.show()

### Cortar sinais

In [ ]:
def show_cut_signals(signals):
    i=0
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 6))
    axes = [ax1,ax2,ax3]

    for stream in signals:
        if i <3:
            channel = stream['Channel']
            date = stream['Date']

            new = DH.cutSignal(stream)
            x = new['X']
            y = new['Y']

            print(len(x),' : ',len(y))

            axes[i].plot(x, y, label=f"{channel}_raw")
            axes[i].set_title(f"{channel} - Raw Signal")
            axes[i].set_xlabel("Time (s)")
            axes[i].set_ylabel("LFP (µV)")
            axes[i].legend()

            i+=1

    plt.tight_layout()
    plt.show()


### Correr todos os ficheiros

In [ ]:
def run_json_files(directory):
    """Recursively find and execute JSON files."""
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".json"):
                json_path = os.path.join(root, file)
                try:
                    with open(json_path, "r", encoding="utf-8") as f:
                        
                        Data = DH.getData(json_path)
                        print("--------------")                     
                        if 'Impedance' in Data['Device'].keys() and Data['Device']['Impedance']:
                            print(json_path)
                            print(Data['Device']['SessionDate'])
                            print(Data['Device']['Impedance'][0].keys())

                        elif Data['Device']['Impedance'] == []:
                            print(json_path) 
                            print('NOOOO:', Data['Device']['SessionDate'])
                                
                except Exception as e:
                    print(f"Error processing {json_path}: {e}")


# O que é Impedance/Most Recent Signal ??

In [ ]:
for recent in Data['Setup']['MostRecentInSessionSignalCheck']:
    print(recent['Channel'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))  # 1 row, 2 columns

# Plot raw signal in the first subplot
ax1.plot(x, y, label=f"{channel}_raw")
ax1.set_title(f"{channel} - Raw Signal")
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("LFP (µV)")
ax1.legend()

# Plot cleaned signal in the second subplot
ax2.plot(new_x, new_y, label=f"{channel}_cleaned", color='orange')
ax2.set_title(f"{channel} - Cleaned Signal")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("LFP (µV)")
ax2.legend()

# Adjust layout
plt.tight_layout()
plt.show()


file_path = "C:\\Users\\OCD\\Desktop\\DATA\\RN\\2024-12-19\\Report_Json_Session_Report_20250212T121933.json"

#Setup: Impedance + MostRecentInSessionSignalCheck
#Survey: Impedance only

Data = DH.getData(file_path)

print(Data['Device']['Impedance' ][-1]['Hemisphere'][1]['SessionImpedance'].keys())


#plt.show()

plt.figure()
for rec in recs:
    print(rec)
    x = rec['X']
    y = rec['Y']
    plt.plot(x,y,label = rec['Channel'])

plt.legend()

recent = Data['Device']['MostRecentInSessionSignalCheck'] #PSD - Survey da sessão 
if recent:
    plt.figure()
    for stream in recent:
        channel = stream['Channel']
        x = stream['SignalFrequencies']
        y = stream['SignalPsdValues']
        peak_x = stream['PeakFrequencies']
        peak_y = stream['PeakValues']
        plt.plot(x,y,label=utl.after_point(channel))
        plt.scatter(peak_x,peak_y)

    plt.legend()

survey = DH.getSignals(Data,'LFPTimeDomain')

if survey:
    plt.figure()
    for stream in survey:
        channel = stream['Channel']
        x = stream['SignalFrequencies']
        y = stream['SignalPsdValues']
        peak_x = stream['PeakFrequencies']
        peak_y = stream['PeakValues']
        plt.plot(x,y,label=utl.after_point(channel))
        plt.scatter(peak_x,peak_y)

    plt.legend()
    
plt.show()

### BRAVO CHECK MISSING DATA

In [ ]:
mine = DH.checkMissingData(Data)

print('----------MINE--------------')
for m in mine: print(m)

plt.figure()
plt.plot(time,signal)
plt.show()

missing = DH.checkMissingData(Data)

def unwrap(x, cap=None):
    if cap is None:
        cap = np.max(x) + 1
    
    unwrappedX = np.array(range(len(x)))
    unwrappedX[0] = x[0]
    currentRolls = 0
    for n in range(1,len(x)):
        if x[n] < x[n-1]:
            currentRolls += cap
        unwrappedX[n] = x[n] + currentRolls
    
    return unwrappedX

def checkMissingPackage(Data):
    if "IndefiniteStreaming" in Data['Indefinite'].keys():
        Data = Data['Indefinite']
        for nStream in range(len(Data["IndefiniteStreaming"])):
            TDSequences = unwrap(utl.string2numbers(Data["IndefiniteStreaming"][nStream]["GlobalSequences"][1:-1]), cap=256) #unwrap?
            raw_len = len(Data["IndefiniteStreaming"][nStream]["TimeDomainData"])
            print(f"{nStream} | TD: {raw_len} | GS: {TDSequences[-1]} ")
            missingSequence = list()
            for n in range(1,len(TDSequences)):
                jumppedSequence = TDSequences[n]-TDSequences[n-1]
                if jumppedSequence > 1:
                    print(nStream, ':', TDSequences[n-1], '-->', TDSequences[n])
                    print(TDSequences[n-1:])
                    missingIndexes = np.array(range(1, jumppedSequence)) + TDSequences[n-1]
                    missingSequence.extend(missingIndexes)
                    #print(f"Sequence {n}: {missingIndexes}")
                    
            Data["IndefiniteStreaming"][nStream]["Missing"] = np.zeros(len(Data["IndefiniteStreaming"][nStream]['TimeDomainData']))
            PacketSize = int(np.mean(utl.string2numbers(Data["IndefiniteStreaming"][nStream]["GlobalSequences"][1:-1])))
            if len(missingSequence) > 0:
            #for nMissing in missingSequence:
            #    insertionIndex = np.where(TDSequences < nMissing)[0][-1] + 1
                startIndex = int(np.sum(utl.string2numbers(Data["IndefiniteStreaming"][nStream]["GlobalSequences"][1:-1])))
            #    TDSequences = np.concatenate((TDSequences[:insertionIndex], [nMissing], TDSequences[insertionIndex:]))
            #    Data["IndefiniteStreaming"][nStream]["GlobalPacketSizes"] = np.concatenate((utl.string2numbers(Data["IndefiniteStreaming"][nStream]["GlobalSequences"][1:-1])[:insertionIndex], [PacketSize], TDSequences[insertionIndex:]))
            #    Data["IndefiniteStreaming"][nStream]["Data"] = np.concatenate((Data["IndefiniteStreaming"][nStream]["TimeDomainData"][:startIndex],np.zeros((PacketSize)),Data["IndefiniteStreaming"][nStream]["TimeDomainData"][startIndex:]))
                Data["IndefiniteStreaming"][nStream]["Missing"] = np.concatenate((Data["IndefiniteStreaming"][nStream]["Missing"][:startIndex],np.ones((PacketSize)),Data["IndefiniteStreaming"][nStream]["Missing"][startIndex:]))
                #Data["IndefiniteStreaming"][nStream]["Time"] = np.array(range(len(Data["IndefiniteStreaming"][nStream]["TimeDomainData"]))) / Data["IndefiniteStreaming"][nStream]["SamplingRateinHz"]
                #print(f"Warning: Missing sequence occured for Stream #{nStream}, Data insertion complete. Check ['Missing'] field.")
    ###       
    if "StreamingTD" in Data.keys() and "StreamingPower" in Data.keys():
        if len(Data["StreamingTD"]) != len(Data["StreamingPower"]):
            raise Warning("Number of stream in Time-domain does not match number of stream in Power channel")
            
        for nStream in range(len(Data["StreamingTD"])):
            Data["StreamingTD"][nStream]["Missing"] = np.zeros(Data["StreamingTD"][nStream]["Data"].shape)
            Data["StreamingPower"][nStream]["Missing"] = np.zeros((Data["StreamingPower"][nStream]["Power"].shape[0]))
            
            # TicksInMs
            ChangesInMs = np.diff(Data["StreamingTD"][nStream]["Ticks"])
            TimePerPackage = np.median(ChangesInMs)
            if len(np.where(ChangesInMs < 0)[0]) > 0:
                print("TicksInMs Revamped")
                raise Exception("Bad Format in TicksInMs")
            
            TimePerPacket = np.median(ChangesInMs)
            MissingPacket = np.where(ChangesInMs > TimePerPacket)[0] + 1
            TDSequences = np.arange(len(Data["StreamingTD"][nStream]["Ticks"]))
            
            # is all missing sequence accounted for?
            PacketSize = int(np.mean(Data["StreamingTD"][nStream]["GlobalPacketSizes"]))
            if len(MissingPacket) > 0:
                for missingIndex in MissingPacket:
                    if not ChangesInMs[missingIndex-1] % TimePerPacket == 0:
                        print(f"TicksInMs Reversed for {nStream}")
                        #raise Exception("Time Skip is not full package drop")
                        
                    numMissingPacket = int(ChangesInMs[missingIndex-1] / TimePerPacket - 1)
                    insertionIndex = np.where(TDSequences < missingIndex)[0][-1] + 1
                    startIndex = int(np.sum(Data["StreamingTD"][nStream]["GlobalPacketSizes"][:insertionIndex]))
                    TDSequences = np.concatenate((TDSequences[:insertionIndex], np.zeros(numMissingPacket), TDSequences[insertionIndex:]))
                    Data["StreamingTD"][nStream]["GlobalPacketSizes"] = np.concatenate((Data["StreamingTD"][nStream]["GlobalPacketSizes"][:insertionIndex], PacketSize*np.ones(numMissingPacket), Data["StreamingTD"][nStream]["GlobalPacketSizes"][insertionIndex:]))
                    Data["StreamingTD"][nStream]["Data"] = np.concatenate((Data["StreamingTD"][nStream]["Data"][:startIndex],np.zeros((PacketSize*numMissingPacket)),Data["StreamingTD"][nStream]["Data"][startIndex:]))
                    Data["StreamingTD"][nStream]["Missing"] = np.concatenate((Data["StreamingTD"][nStream]["Missing"][:startIndex],np.ones((PacketSize*numMissingPacket)),Data["StreamingTD"][nStream]["Missing"][startIndex:]))
                Data["StreamingTD"][nStream]["Time"] = np.array(range(len(Data["StreamingTD"][nStream]["Data"]))) / Data["StreamingTD"][nStream]["SamplingRate"]
                #print(f"Warning: Missing sequence occured for Stream #{nStream}, Data insertion complete. Check ['Missing'] field.")
            
            # Check Power Channel Now
            ChangesInMs = np.around(np.diff(Data["StreamingPower"][nStream]["Time"]),3)
            if len(np.where(ChangesInMs < 0)[0]) > 0:
                print("TicksInMs Reversed")
                raise Exception("Bad Format in TicksInMs for Power Channel")
            
            TimePerPacket = np.percentile(ChangesInMs,5)
            MissingPacket = np.where(ChangesInMs > TimePerPacket)[0] + 1
            
            if len(MissingPacket) > 0:
                newTimestamp = np.arange(Data["StreamingPower"][nStream]["Time"][0], Data["StreamingPower"][nStream]["Time"][-1]+TimePerPacket, TimePerPacket)
                processedPower = np.zeros((len(newTimestamp),2))
                processedStimulation = np.zeros((len(newTimestamp),2))
                Data["StreamingPower"][nStream]["Missing"] = np.zeros((len(newTimestamp)))
                for i in range(2):
                    processedPower[:,i] = np.interp(newTimestamp, Data["StreamingPower"][nStream]["Time"], Data["StreamingPower"][nStream]["Power"][:,i])
                    processedStimulation[:,i] = np.interp(newTimestamp, Data["StreamingPower"][nStream]["Time"], Data["StreamingPower"][nStream]["Stimulation"][:,i])
                
                for t in range(len(newTimestamp)):
                    if not newTimestamp[t] in Data["StreamingPower"][nStream]["Time"]:
                        Data["StreamingPower"][nStream]["Missing"][t] = 1
                        
                Data["StreamingPower"][nStream]["Power"] = processedPower
                Data["StreamingPower"][nStream]["Stimulation"] = processedStimulation
                Data["StreamingPower"][nStream]["Time"] = newTimestamp
                
            # Compensate for shifting less than 500ms, which will be indicated by the TickInMs
            TimeShift = Data["StreamingPower"][nStream]["InitialTickInMs"] - Data["StreamingTD"][nStream]["Ticks"][0] / 1000
            if TimeShift < -1000:
                TimeShift = np.round(TimeShift+3276.8,2)
            elif TimeShift > 1000:
                TimeShift = np.round(TimeShift-3276.8,2)
            else:
                TimeShift = np.round(TimeShift,2)
            Data["StreamingPower"][nStream]["Time"] += TimeShift
            #print(f"Warning: Missing sequence occured for Stream #{nStream}, Data insertion complete. Check ['Missing'] field.")
            ###
    

#bravo = checkMissingPackage(Data)

plt.figure()
for i in range(len(Data['Indefinite']['IndefiniteStreaming'])):
    seq = utl.string2numbers(Data['Indefinite']['IndefiniteStreaming'][i]['GlobalSequences'][1:-1])
    plt.plot(seq,'o')
    plt.legend()

plt.show()

### encontrar coordenadas eventos streaming

In [ ]:
streaming = DH.getSignals(Data, 'Streaming')

file = 'streaming_events.txt'

events = utl.read_event_file(file)

events_times = []

plt.figure()
for stream in streaming:
    date = stream['Date']
    x= stream['X']
    y = stream['Y']
    new_time = [date + timedelta(seconds=val) for val in stream['X']]
    
    for event in events:
        hour, minute, type = event
        
        idx = utl.find_closest_index(new_time,hour,minute)
        if idx != None: 
            events_times.append([idx, type])

    plt.plot(new_time,y,label=date)

    if events_times:
        for time, type in events_times:
            print(time," | ", stream['Y'][time])
            plt.scatter(new_time[time],300,label = type)
        events_times = []

plt.legend()
plt.show()

## Aceder qualquer função scipy com uma só função

In [ ]:
fs = 250/2

def get_functions(module):
    """Retrieve all callable functions from a module."""

    functions = {name: obj for name, obj in inspect.getmembers(module, inspect.isfunction)}
    return functions

def call_function2(module, func_name, *args, **kwargs):
    """Call a function from a module by name, print its documentation, and handle exceptions."""
    functions = get_functions(module)
    
    if func_name not in functions:
        print(f"Function '{func_name}' not found in {module.__name__}")
        return None

    func = functions[func_name]

    # Print function documentation
    print(f"\nDocumentation for '{func_name}':\n")
    #print(func.__doc__ if func.__doc__ else "No documentation available.")
    print("\n" + "="*80 + "\n")

    # Get function signature
    sig = inspect.signature(func)
    
    # Extract required parameters (no defaults)
    required_params = [
        param.name for param in sig.parameters.values()
        if param.default is inspect.Parameter.empty and param.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.POSITIONAL_ONLY)
    ]
    
    # Extract keyword default parameters
    default_kwargs = {
        param.name: param.default for param in sig.parameters.values()
        if param.default is not inspect.Parameter.empty and param.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)
    }

    try:
        return func(*args, **kwargs)
    except Exception as e:
        print(f"Error calling {func_name}: {e}")

        if args:
            print("Error occurred even with provided arguments. Not retrying.")
            return None  # If the user provided arguments, don't retry

        if required_params:
            print(f"Cannot retry because required positional arguments are missing: {required_params}")
            return None  # Can't call the function without required positional args
        
        if default_kwargs:
            print("Retrying with default keyword arguments...")
            try:
                return func(**default_kwargs)
            except Exception as e2:
                print(f"Error with default parameters as well: {e2}")
                return None
        else:
            print("No default parameters available.")
            return None

x = stream['X'][:1000]
y = stream['Y'][:1000]

# Example usage
functions = get_functions(scipy.signal)
#want = input(f"Available functions: {list(functions.keys())}")

# Calling a function (e.g., butterworth filter design)
b, a = call_function(scipy.signal, "butter", 4, 30, "low", fs = fs)

filtered = call_function(scipy.signal, "filtfilt", b,a, y)

#b, a = ss.butter(4, 30, "low", fs=fs)
#filtered = ss.filtfilt(b, a, y)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))



ax1.plot(x,y,label = 'Raw')
ax2.plot(x, filtered,label = 'Filtered')

ax1.legend()
ax2.legend()

plt.legend()
plt.show()

### Guardar plot (screenshot)

In [ ]:
import dearpygui.dearpygui as dpg
from PIL import ImageGrab
import time

# Create a context
dpg.create_context()

# Create a window with a plot
with dpg.window(label="Plot Window", tag = "main_window"):
    with dpg.plot(label="Example Plot", height=400, width=600, tag="my_plot"):
        dpg.add_plot_axis(dpg.mvXAxis, label="X Axis", tag="x_axis")
        dpg.add_plot_axis(dpg.mvYAxis, label="Y Axis", tag="y_axis")

        x_data = [1, 2, 3, 4, 5]
        y_data = [10, 20, 15, 30, 25]

        dpg.add_line_series(x_data, y_data, parent="y_axis", label="Line")

# Function to capture only the plot
def save_plot(sender, app_data, user_data):
    plot_tag = user_data
    time.sleep(1)  # Give time for rendering

    # Get plot's position and size
    min_x, min_y = dpg.get_item_rect_min(plot_tag)  # Top-left corner
    max_x, max_y = dpg.get_item_rect_max(plot_tag)  # Bottom-right corner

    # Get viewport position (absolute position on screen)
    vp_x, vp_y = dpg.get_viewport_pos()

    # Calculate the absolute screen position of the plot
    abs_min_x = min_x + vp_x 
    abs_min_y = min_y + vp_y +4
    abs_max_x = max_x + vp_x +5
    abs_max_y = max_y + vp_y +5

    bbox = (abs_min_x, abs_min_y, abs_max_x, abs_max_y)
    
    # Capture only the plot area
    screenshot = ImageGrab.grab(bbox=bbox)

    screenshot.save(f"plots\\{plot_tag}_image.png")  # Save the image
    print(f"Plot saved as '{plot_tag}_image.png'")

# Button to trigger save
with dpg.window(label="Controls", pos=(650, 100)):
    dpg.add_button(label="Save Plot", callback=save_plot, user_data='my_plot')


# Create and show viewport
dpg.create_viewport(title="Dear PyGui Plot Save", width=800, height=600)
dpg.setup_dearpygui()
dpg.show_viewport()

dpg.start_dearpygui()
dpg.destroy_context()


'''
    screenshot.save(f"plots\\{plot_tag}_image.png")  # Save the image
    print(f"Plot saved as '{plot_tag}_image.png'")

# Button to trigger save
with dpg.window(label="Controls", pos=(650, 100)):
    dpg.add_button(label="Save Plot", callback=save_plot, user_data='my_plot')

'''

### Melhorado

In [ ]:
import dearpygui.dearpygui as dpg
import Preprocessing as pre
import scipy.signal as ss
import DataHandling as DH
import numpy as np

file_path = "C:\\Users\\OCD\\Desktop\\DATA\\RN\\2024-10-14\\Report_Json_Session_Report_20241125T162143.json"
Data = DH.getData(file_path)
stream = DH.getSignals(Data, 'Streaming')[0]

iir_filter_design_functions = ['butter','buttord','cheby1','cheb1ord','cheby2','cheb2ord','ellip',
                               'ellipord','bessel','iirnotch','iirpeak', 'iircomb']

'''all_functions = pre.get_functions(ss)
iir_filter_design_functions = list(all_functions.keys())
'''
# Dictionary to store user inputs
parameter_values = {}
input_fields = {}  # Store input field IDs


dpg.create_context()

with dpg.window(label="IIR Filter Designer", width=600, height=500):
    dpg.add_text("Select a filter function:")
    param_container = dpg.add_child_window(width=-1, height=300)  # Space to add inputs dynamically

    def create_input_fields(sender, app_data):
        """Dynamically create input fields for selected function."""
        global parameter_values, input_fields
        
        dpg.delete_item(param_container, children_only=True)  # Clear previous inputs
        params = pre.get_required_params(ss, app_data)
        
        if not params:
            dpg.add_text("No parameters found", parent=param_container)
            return
        dpg.add_text("Required parameters: ", parent=param_container)
        parameter_values.clear()  # Reset parameter storage
        input_fields.clear()  # Reset input field storage

        for param, default in params.items():
            input_id = dpg.add_input_text(
                label=param, 
                default_value=str(default) if default is not None else "",
                parent=param_container
            )
            input_fields[param] = input_id  # Store input field reference

        # Add save button
        dpg.add_button(label="Save Parameters", callback=save_parameters, parent=param_container, user_data = app_data)

    def save_parameters(sender, app_data, user_data):
        """Saves input values to the dictionary and prints them."""
        global parameter_values
        parameter_values.clear()  # Reset before saving

        for param, input_id in input_fields.items():
            value = dpg.get_value(input_id)
            parameter_values[param] = value

        print("Saved Parameters:", parameter_values)
        dpg.add_button(label="Plot filtered signal", callback=apply_filt, parent=param_container, user_data=user_data)

    def apply_filt(sender, app_data, user_data):
        func = user_data
        print(func)
        
        # Assuming call_function takes the filter function and the user-provided parameters
        args = pre.call_function(ss, func, parameter_values)
        x = stream['X']
        y = stream['Y']

        filtered = pre.call_function2(ss, 'filtfilt', *args, y)

        # Use the main thread for plotting
        plot_filtered_signal(x, y, filtered)

    def plot_filtered_signal(x, y, filtered):
        with dpg.plot(label=f"Signals", height=400, width=600, parent =param_container ):
            dpg.add_plot_legend()
            dpg.add_plot_axis(dpg.mvXAxis, label="Frequency (Hz)", tag="x_axis")
            dpg.add_plot_axis(dpg.mvYAxis, label="LFP Power (dB)", tag="y_axis")
            print(f"{len(x)} | {len(y)} | {len(filtered)}")
            dpg.add_line_series(x, y, label="Raw", parent="y_axis")
            dpg.add_line_series(x, np.ascontiguousarray(filtered), label=f"Filtered", parent="y_axis")

            
    # Dropdown list for filters
    dpg.add_listbox(
        items=iir_filter_design_functions, 
        label="Filters", 
        callback=create_input_fields
    )

dpg.create_viewport(title='Temporario', width=800, height=600)
dpg.setup_dearpygui()
dpg.show_viewport()
dpg.start_dearpygui()
dpg.destroy_context()



### Para feature extraction


In [ ]:
import scipy.signal as ss
import Preprocessing as pre
import dearpygui.dearpygui as dpg
import DataHandling as DH
import numpy as np

''' For feature extraction - get all scipy functions'''

#Spectral analysis
spectral_analysis_functions = ['periodogram','welch','coherence','spectrogram','stft']

file_path = "C:\\Users\\OCD\\Desktop\\DATA\\RN\\2024-10-14\\Report_Json_Session_Report_20241125T162143.json"
Data = DH.getData(file_path)
stream = DH.getSignals(Data, 'Streaming')[0]


# Dictionary to store user inputs
parameter_values = {}
input_fields = {}  # Store input field IDs


dpg.create_context()

with dpg.window(label="IIR Filter Designer", width=600, height=500):
    dpg.add_text("Select a filter function:")
    param_container = dpg.add_child_window(width=-1, height=300)  # Space to add inputs dynamically

    def create_input_fields(sender, app_data):
        """Dynamically create input fields for selected function."""
        global parameter_values, input_fields
        
        dpg.delete_item(param_container, children_only=True)  # Clear previous inputs
        params = pre.get_required_params(ss, app_data)
        
        if not params:
            dpg.add_text("No parameters found", parent=param_container)
            return
        dpg.add_text("Required parameters: ", parent=param_container)
        parameter_values.clear()  # Reset parameter storage
        input_fields.clear()  # Reset input field storage

        for param, default in params.items():
            input_id = dpg.add_input_text(
                label=param, 
                default_value=str(default) if default is not None else "",
                parent=param_container
            )
            input_fields[param] = input_id  # Store input field reference

        # Add save button
        dpg.add_button(label="Save Parameters", callback=save_parameters, parent=param_container, user_data = app_data)

    def save_parameters(sender, app_data, user_data):
        """Saves input values to the dictionary and prints them."""
        global parameter_values
        parameter_values.clear()  # Reset before saving

        for param, input_id in input_fields.items():
            value = dpg.get_value(input_id)
            parameter_values[param] = value

        print("Saved Parameters:", parameter_values)
        dpg.add_button(label="Plot filtered signal", callback=apply_filt, parent=param_container, user_data=user_data)

    def apply_filt(sender, app_data, user_data):
        func = user_data

        x = stream['X']
        y = stream['Y']

        parameter_values['x'] = y
        
        # Assuming call_function takes the filter function and the user-provided parameters
        signal = pre.call_function(ss, func, parameter_values)
        if signal:
            plot_filtered_signal(x, y, signal)
        else: print(signal)

    def plot_filtered_signal(x, y, filtered):
        with dpg.plot(label=f"Signals", height=400, width=600, parent =param_container ):
            dpg.add_plot_legend()
            dpg.add_plot_axis(dpg.mvXAxis, label="Time (s)", tag="x_axis")
            dpg.add_plot_axis(dpg.mvYAxis, label="LFP amp (microV)", tag="y_axis")
            dpg.add_line_series(x, y, label="Raw", parent="y_axis")
        with dpg.plot(label=f"Spectral Analysis", height=400, width=600, parent =param_container ):
            dpg.add_plot_legend()
            dpg.add_plot_axis(dpg.mvXAxis, label="Frequency (Hz)", tag="xx")
            dpg.add_plot_axis(dpg.mvYAxis, label="LFP Power (dB)", tag="yy")
            dpg.add_line_series(np.ascontiguousarray(filtered[0]),np.ascontiguousarray(filtered[1]), label=f"Filtered", parent="yy")

            
    # Dropdown list for filters
    dpg.add_listbox(
        items=spectral_analysis_functions, 
        label="Filters", 
        callback=create_input_fields
    )

dpg.create_viewport(title='Temporario', width=800, height=600)
dpg.setup_dearpygui()
dpg.show_viewport()
dpg.start_dearpygui()
dpg.destroy_context()

# FILTROS

In [ ]:
# Hilbert Transform
import scipy.signal as ss
import numpy as np

filtered = ss.hilbert(y[:250])
fs = 250
amplitude_envelope = np.abs(filtered)
instantaneous_phase = np.unwrap(np.angle(filtered))
instantaneous_frequency = np.diff(instantaneous_phase) / (2.0*np.pi) * fs

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
'''
ax1.plot(x,y,label = 'Raw')
ax2.plot(x,filtered,label = 'Median Filter')

ax1.legend()
ax2.legend()'''

ax1.plot(x[:250],y[:250],label = 'Raw')
ax1.plot(x[:250],amplitude_envelope,label = 'Hilbert Envelope', linestyle = '--')
ax2.plot(x[1:250],instantaneous_frequency,label = 'Hilbert Instantaneoys Phase', linestyle = '--')

ax1.legend()
ax2.legend()

plt.legend()
plt.show()

In [ ]:
# Decimate, by 10

filtered = ss.decimate([x,y],10)
fs = 250



fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))


ax1.plot(x,y,label = 'Raw')
ax2.plot(filtered[0],filtered[1],label = 'Decimate')

ax1.legend()
ax2.legend()

plt.legend()
plt.show()